# Student Performance EDA — Mixed Encoding (Ordinal + One-Hot)

This notebook evaluates a `LinearRegression` model on `Exam_Score` using a **mixed encoding strategy**:

- **Ordinal encoding** for ordered categories (`Low / Medium / High`, `Near / Moderate / Far`)
- **One-hot encoding** for nominal categories (gender, school type, etc.)

This respects the natural order of some features, which can help a linear model pick up the right direction of effect.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score, train_test_split

In [2]:
# Load the raw dataset and inspect missing values.
data_path = Path("../data/StudentPerformanceFactors.csv")
raw_df = pd.read_csv(data_path)

print("Raw shape:", raw_df.shape)
print("Missing values:")
print(raw_df.isna().sum()[raw_df.isna().sum() > 0])

Raw shape: (6607, 20)
Missing values:
Teacher_Quality             78
Parental_Education_Level    90
Distance_from_Home          67
dtype: int64


In [3]:
# Drop rows with missing values to build a clean base dataset.
base_df = raw_df.dropna().copy()
print("Clean shape after dropna():", base_df.shape)

Clean shape after dropna(): (6378, 20)


## Step 1 — Ordinal Encoding for Ordered Categories

Columns where the values have a clear order are mapped to integers.
This preserves the distance between levels (Low < Medium < High) instead of treating them as independent binary flags.

In [4]:
mixed_df = base_df.copy()

ordinal_mappings = {
    "Parental_Involvement": {"Low": 1, "Medium": 2, "High": 3},
    "Access_to_Resources": {"Low": 1, "Medium": 2, "High": 3},
    "Motivation_Level": {"Low": 1, "Medium": 2, "High": 3},
    "Family_Income": {"Low": 1, "Medium": 2, "High": 3},
    "Teacher_Quality": {"Low": 1, "Medium": 2, "High": 3},
    "Distance_from_Home": {"Near": 1, "Moderate": 2, "Far": 3},
}

for column, mapping in ordinal_mappings.items():
    mixed_df[column] = mixed_df[column].map(mapping)

print("Ordinal columns after mapping:")
print(mixed_df[list(ordinal_mappings.keys())].head())

Ordinal columns after mapping:
   Parental_Involvement  Access_to_Resources  Motivation_Level  Family_Income  \
0                     1                    3                 1              1   
1                     1                    2                 1              2   
2                     2                    2                 2              2   
3                     1                    2                 2              2   
4                     2                    2                 2              2   

   Teacher_Quality  Distance_from_Home  
0                2                   1  
1                2                   2  
2                2                   1  
3                2                   2  
4                3                   1  


## Step 2 — One-Hot Encoding for Nominal Categories

The remaining categorical columns have no meaningful order, so they are one-hot encoded.

In [5]:
nominal_columns = [
    "Extracurricular_Activities",
    "Internet_Access",
    "Learning_Disabilities",
    "Gender",
    "School_Type",
    "Peer_Influence",
    "Parental_Education_Level",
]

mixed_df = pd.get_dummies(mixed_df, columns=nominal_columns, drop_first=True)
print("Shape after mixed encoding:", mixed_df.shape)

Shape after mixed encoding: (6378, 22)


In [6]:
X = mixed_df.drop(columns=["Exam_Score"])
y = mixed_df["Exam_Score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

test_r2 = r2_score(y_test, model.predict(X_test))
cv_scores = cross_val_score(model, X, y, cv=5, scoring="r2")

print(f"Test R2:      {test_r2:.4f}")
print(f"CV Mean R2:   {cv_scores.mean():.4f}")
print(f"CV Std:       {cv_scores.std():.4f}")

Test R2:      0.7324
CV Mean R2:   0.7207
CV Std:       0.0695


## Conclusion

| Metric | Value |
|--------|-------|
| Test R2 | 0.7324 |
| CV Mean R2 | 0.7207 |
| CV Std | 0.0695 |

Mixed encoding slightly outperforms the all-dummies version (see `01_eda_one_hot.ipynb`).
The margin is very small, which suggests the linear model already captures most of the ordinal structure even with one-hot encoding.

The key takeaway is that the dataset is close to linearly structured — consistent with `LinearRegression` outperforming tree-based ensembles in the broader model comparison.